In [ ]:
# Import all necessary packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import distinctipy
import matplotlib.colors as mcolors
from plottable import ColumnDefinition, Table
import rootutils

# Set directories and load data
path_root = str(rootutils.find_root(indicator=".project-root"))
path_plots = f"{path_root}/plots"
path_data = f"{path_root}/data/age-regression"
df_feats = pd.read_excel(f"{path_data}/features.xlsx", index_col=0)
imms = df_feats.index.to_list()
df = pd.read_excel(f"{path_data}/data.xlsx")

# Prepare colors
def make_rgb_transparent(rgb, bg_rgb, alpha):
    return [alpha * c1 + (1 - alpha) * c2 for (c1, c2) in zip(rgb, bg_rgb)]

# Prepare data for the figure
imms_all = pd.read_excel(f"{path_root}/data/cytokines-regression/features.xlsx").columns.to_list()
feats_dict_log = {x: f"{x}_log" for x in imms}

sns.set_theme(style='ticks')
fig = plt.figure(layout='constrained', figsize=(15, 10))
subfigs = fig.subfigures(1, 2, width_ratios=[4, 9], wspace=0.05)

axs = subfigs[0].subplots(4, 1, height_ratios=[0.2, 0.2, 0.8, 0.58], gridspec_kw={'wspace':0.25, 'hspace': 0.05}, sharey=False, sharex=False)

df_fig = df.loc[df['Status'] == 'Control', ['Age', 'Status', 'Split', 'EpInflammAge']]
df_fig['Error'] = df_fig['EpInflammAge'] - df_fig['Age']
df_metrics = pd.read_excel(f"{path_root}/models/EpInflammAge/metrics.xlsx", index_col=0)

row_id_table = 0
row_id_hist = 1
row_id_scatter = 2
row_id_violin = 3

# Table in Figure 4a
df_table = pd.DataFrame(index=['MAE', "Pearson's R", "Bias"], columns=['Train', 'Validation', 'Test', 'Total'])
for part in ['Train', 'Validation', 'Test', 'Total']:
    df_table.at['MAE', part] = f"{df_metrics.at[part, 'MAE']:0.3f}"
    df_table.at["Pearson's R", part] = f"{df_metrics.at[part, 'Rho']:0.3f}"
    df_table.at["Bias", part] = f"{df_metrics.at[part, 'Bias']:0.3f}"

col_defs = [
    ColumnDefinition(
        name="index",
        title='',
        textprops={"ha": "center", "weight": "bold"},
        width=2.5,
    ),
    ColumnDefinition(
        name="Train",
        textprops={"ha": "left"},
        width=1.5,
        border="left"
    ),
    ColumnDefinition(
        name="Validation",
        textprops={"ha": "left"},
        width=2.2,
    ),
    ColumnDefinition(
        name="Test",
        textprops={"ha": "left"},
        width=1.5,
    ),
    ColumnDefinition(
        name="Total",
        textprops={"ha": "left"},
        width=1.5,
    )
]

axs[row_id_table].text(-2, -1, 'A', fontsize=30, fontfamily='arial')

table = Table(
    df_table,
    column_definitions=col_defs,
    row_dividers=True,
    footer_divider=False,
    ax=axs[row_id_table],
    textprops={"fontsize": 8},
    row_divider_kw={"linewidth": 1, "linestyle": (0, (1, 1))},
    col_label_divider_kw={"linewidth": 1, "linestyle": "-"},
    column_border_kw={"linewidth": 1, "linestyle": "-"},
).autoset_fontcolors(colnames=['Train', 'Validation', 'Test', 'Total'])

# Histogram in Figure 4a
hist_bins = np.linspace(0, 120, 25)
histplot = sns.histplot(
    data=df_fig,
    bins=hist_bins,
    edgecolor='k',
    linewidth=1,
    x="Age",
    color='lightslategray',
    ax=axs[row_id_hist]
)
axs[row_id_hist].set_xticks([])
axs[row_id_hist].set_xlim(0, 105)
axs[row_id_hist].set_ylabel("Count")
axs[row_id_hist].set_xlabel("")

# KDE and scatter in Figure 4a
kdeplot = sns.kdeplot(
    data=df_fig.loc[df_fig['Split'].isin(['Train', 'Validation']), :],
    x='Age',
    y='EpInflammAge',
    fill=True,
    cbar=False,
    thresh=0.005,
    color=make_rgb_transparent(mcolors.hex2color(mcolors.CSS4_COLORS['lightslategray']), (1, 1, 1), 0.25),
    cut=0,
    legend=False,
    ax=axs[row_id_scatter]
)
scatter = sns.scatterplot(
    data=df_fig.loc[df_fig['Split'] == 'Test', :],
    x='Age',
    y="EpInflammAge",
    linewidth=0.01,
    alpha=0.8,
    edgecolor="k",
    s=2,
    color='lightslategray',
    ax=axs[row_id_scatter],
)
axs[row_id_scatter].axline((0, 0), slope=1, color="black", linestyle=":")
axs[row_id_scatter].set_xlim(0, 105)
axs[row_id_scatter].set_ylim(0, 105)
axs[row_id_scatter].set_ylabel("Prediction")
axs[row_id_scatter].set_xlabel("Age")

# Violins in Figure 4a
violin = sns.violinplot(
    data=df_fig,
    x='Split',
    y='Error',
    palette={'Train': 'lightgray', 'Validation': 'lightslategray', 'Test': 'dimgrey'},
    scale='width',
    order=['Train', 'Validation', 'Test'],
    saturation=0.75,
    legend=False,
    ax=axs[row_id_violin]
)
axs[row_id_violin].set_xlabel('')
axs[row_id_violin].set_ylabel('Age Acceleration')

# SHAP values can be different due to randomization. Use previously calculated files for global explainability on test set
ids_shap = df.index[(df['Status'] == 'Control') & ((df['Split'] == 'Test'))].values
df_shap = df.loc[ids_shap, ['Age', 'Status', 'Split', 'EpInflammAge'] + list(imms)].rename(columns=feats_dict_log)
explanation = pd.read_excel(f"{path_data}/explanation.xlsx")
explanation.index = ids_shap

imm_colors = distinctipy.get_colors(n_colors=len(imms_all), exclude_colors=[mcolors.hex2color(mcolors.CSS4_COLORS['gray'])], rng=42)
imm_colors_dict = {}
for imm_id, imm in enumerate(imms_all):
    imm_color = imm_colors[imm_id]
    imm_colors_dict[imm] = imm_color

ds_fi = pd.DataFrame(index=imms, columns=['abs(|SHAP|)'])
for f in imms:
    ds_fi.at[f, 'abs(|SHAP|)'] = explanation[f].abs().mean()
ds_fi.sort_values(['abs(|SHAP|)'], ascending=[False], inplace=True)
ds_fi['Features'] = ds_fi.index.values

axs = subfigs[1].subplots(1, 2, width_ratios=[2, 7], gridspec_kw={'wspace':0.01, 'hspace': 0.05}, sharey=True, sharex=False)
    
axs[0].text(-2, -1, 'B', fontsize=30, fontfamily='arial')

# Barplot in Figure 4b
barplot = sns.barplot(
    data=ds_fi,
    x='abs(|SHAP|)',
    y='Features',
    hue='Features',
    palette=imm_colors_dict,
    edgecolor='black',
    dodge=False,
    ax=axs[0]
)
barplot.legend_.remove()
for container in barplot.containers:
    barplot.bar_label(container, label_type='edge', color='gray', fmt='%0.2f', fontsize=12, padding=3.0)
axs[0].set(xlim=(0, 7))
axs[0].set_ylabel('')
axs[0].set(yticklabels=ds_fi.index.to_list())

# Stripplot in Figure 4b
is_colorbar = False
for f in ds_fi.index:
    
    f_shap_ll = explanation[f].quantile(0.01)
    f_shap_hl = explanation[f].quantile(0.99)
    f_shap_index = explanation.index[(explanation[f] >= f_shap_ll) & (explanation[f] <= f_shap_hl)].values
    
    df_f_vals = df_shap.loc[f_shap_index, :]
    f_vals_ll = df_f_vals[f"{f}_log"].quantile(0.01)
    f_vals_hl = df_f_vals[f"{f}_log"].quantile(0.99)
    f_shap_index = df_f_vals.index[(df_f_vals[f"{f}_log"] >= f_vals_ll) & (df_f_vals[f"{f}_log"] <= f_vals_hl)].values
    
    f_shap = explanation.loc[f_shap_index, f].values
    f_vals = df_shap.loc[f_shap_index, f"{f}_log"].values
    
    f_cmap = sns.color_palette("coolwarm", as_cmap=True)
    f_norm = mcolors.Normalize(vmin=min(f_vals), vmax=max(f_vals)) 
    f_colors = {}
    for cval in f_vals:
        f_colors.update({cval: f_cmap(f_norm(cval))})

    strip = sns.stripplot(
        x=f_shap,
        y=[f]*len(f_shap),
        hue=f_vals,
        palette=f_colors,
        jitter=0.37,
        alpha=0.6,
        edgecolor='gray',
        linewidth=0.0,
        size=2,
        legend=False,
        ax=axs[1],
    )
    
    if not is_colorbar:
        sm = plt.cm.ScalarMappable(cmap=f_cmap, norm=f_norm)
        sm.set_array([])
        cbar = strip.figure.colorbar(sm)
        cbar.set_label('Inflammatory markers', labelpad=-4, fontsize='large')
        cbar.set_ticks([min(f_vals), max(f_vals)])
        cbar.set_ticklabels(["Min", "Max"])
        is_colorbar = True
    
axs[1].set_xlabel('SHAP')

fig.savefig(f"{path_plots}/figure4.png", bbox_inches='tight', dpi=200)
fig.savefig(f"{path_plots}/figure4.pdf", bbox_inches='tight')
plt.close(fig)